In [ ]:
.root <- getwd()
while (!file.exists(file.path(.root, 'CLAUDE.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(ggthemes)
  library(ggrepel)
  library(scales)
  library(patchwork)
})

options(repr.plot.width = 14, repr.plot.height = 8)

BASE_THEME <- theme_fivethirtyeight() +
  theme(
    axis.title        = element_text(size = 14),
    axis.text         = element_text(size = 13),
    panel.background  = element_rect(fill = 'white', color = NA),
    plot.background   = element_rect(fill = 'white', color = NA),
    legend.background = element_rect(fill = 'white', color = NA),
    legend.text       = element_text(size = 12),
    legend.title      = element_text(size = 13),
    strip.text        = element_text(size = 14, face = 'bold'),
    plot.title        = element_text(size = 20, face = 'bold'),
    plot.subtitle     = element_text(size = 13)
  )
theme_set(BASE_THEME)

PITCH_NAMES <- c(
  FF = '4-Seam', SI = 'Sinker', FC = 'Cutter',
  CH = 'Changeup', FS = 'Splitter',
  SL = 'Slider', CU = 'Curveball', ST = 'Sweeper', KC = 'Knuckle-curve'
)
PITCH_COLORS <- c(
  FF = '#d62728', SI = '#ff7f0e', FC = '#e377c2',
  CH = '#2ca02c', FS = '#17becf',
  SL = '#bcbd22', CU = '#1f77b4', ST = '#9467bd', KC = '#8c564b'
)

dir.create('results/plots/applications', recursive = TRUE, showWarnings = FALSE)

In [ ]:
seq_df  <- read_csv('data/viz_cache/app_sequencing.csv',           show_col_types = FALSE)
batter  <- read_csv('data/viz_cache/app_batter_vulnerability.csv', show_col_types = FALSE)
batter_pt <- read_csv('data/viz_cache/app_batter_by_pitch_type.csv', show_col_types = FALSE)

cat(sprintf(
  'Loaded | seq pairs: %d | batters: %d | batter-pitch rows: %d\n',
  nrow(seq_df), nrow(batter), nrow(batter_pt)
))

## Pitch Sequencing Value

In [ ]:
PITCH_ORDER <- c('4-Seam','Sinker','Cutter','Changeup','Splitter',
                 'Slider','Curveball','Sweeper','Knuckle-curve')

# Compute baseline distortion per current pitch type
baseline_by_type <- seq_df |>
  group_by(pitch_name) |>
  summarise(baseline = mean(mean_distortion_share), .groups = 'drop')

seq_lift <- seq_df |>
  left_join(baseline_by_type, by = 'pitch_name') |>
  mutate(
    seq_lift      = mean_distortion_share - baseline,
    prev_pitch_name = factor(prev_pitch_name, levels = PITCH_ORDER),
    pitch_name      = factor(pitch_name,      levels = PITCH_ORDER)
  )

p_heatmap <- ggplot(seq_lift,
                    aes(x = prev_pitch_name, y = pitch_name,
                        fill = seq_lift)) +
  geom_tile(color = 'white', linewidth = 0.4) +
  geom_text(
    aes(label = sprintf('%+.1f%%', seq_lift * 100)),
    size = 3.5, fontface = 'bold',
    color = ifelse(abs(seq_lift$seq_lift) > 0.015, 'white', 'grey30')
  ) +
  scale_fill_gradient2(
    low  = '#2980b9', mid = 'white', high = '#e74c3c',
    midpoint = 0,
    labels = percent_format(accuracy = 0.1),
    name = 'Sequence lift\n(vs pitch-type avg)'
  ) +
  labs(
    title    = 'Pitch Sequencing Value: Distortion Lift Above Pitch-Type Baseline',
    subtitle = paste0(
      'Each cell = mean distortion_share for (prev pitch -> current pitch) minus current pitch-type average\n',
      'Red = sequence boosts distortion  |  Blue = sequence reduces distortion  |  min 50 swings per cell'
    ),
    x = 'Previous pitch type',
    y = 'Current pitch type'
  ) +
  theme(
    axis.text.x    = element_text(angle = 35, hjust = 1),
    legend.position = 'right'
  )

options(repr.plot.width = 13, repr.plot.height = 9)
p_heatmap
ggsave('results/plots/applications/sequencing_heatmap.png', p_heatmap,
       width = 13, height = 9, dpi = 150)

In [ ]:
top_seqs <- seq_lift |>
  filter(n >= 200) |>
  mutate(label = paste(prev_pitch_name, '->', pitch_name)) |>
  arrange(desc(seq_lift)) |>
  slice(c(1:10, (n() - 7):n()))

p_top <- ggplot(top_seqs,
                aes(x = reorder(label, seq_lift),
                    y = seq_lift,
                    fill = seq_lift > 0)) +
  geom_col(width = 0.7) +
  geom_hline(yintercept = 0, linewidth = 0.8) +
  geom_text(
    aes(label = sprintf('%+.1f%%', seq_lift * 100),
        hjust = ifelse(seq_lift > 0, -0.15, 1.15)),
    size = 4.2, fontface = 'bold'
  ) +
  scale_fill_manual(values = c('FALSE' = '#2980b9', 'TRUE' = '#e74c3c'), guide = 'none') +
  scale_y_continuous(labels = percent_format(accuracy = 0.1),
                     expand = expansion(mult = c(0.2, 0.2))) +
  coord_flip() +
  labs(
    title    = 'Best and Worst Sequences for Distortion',
    subtitle = 'Distortion share lift above pitch-type average  |  min 200 swings per cell',
    x = NULL, y = 'Sequence lift in distortion share'
  )

options(repr.plot.width = 12, repr.plot.height = 8)
p_top
ggsave('results/plots/applications/sequencing_top.png', p_top,
       width = 12, height = 8, dpi = 150)

## Batter Vulnerability

In [ ]:
top_bottom_batters <- bind_rows(
  batter |> arrange(desc(mean_distortion_share)) |> slice(1:20),
  batter |> arrange(mean_distortion_share)       |> slice(1:10)
) |>
  mutate(
    batter_full_name = str_to_title(batter_full_name),
    is_vulnerable    = mean_distortion_share > median(batter$mean_distortion_share)
  )

p_batter <- ggplot(top_bottom_batters,
                   aes(x = reorder(batter_full_name, mean_distortion_share),
                       y = mean_distortion_share,
                       fill = is_vulnerable)) +
  geom_col(width = 0.7) +
  geom_text(
    aes(label = sprintf('%.1f%%', mean_distortion_share * 100),
        hjust = ifelse(is_vulnerable, -0.15, 1.15)),
    size = 3.8, fontface = 'bold'
  ) +
  scale_fill_manual(values = c('FALSE' = '#2980b9', 'TRUE' = '#e74c3c'), guide = 'none') +
  scale_y_continuous(labels = percent_format(accuracy = 1),
                     expand = expansion(mult = c(0.05, 0.15))) +
  coord_flip() +
  labs(
    title    = 'Batter Vulnerability to Pitcher Distortion',
    subtitle = sprintf(
      'Mean distortion share (distortion_tax / disruption_tax)  |  Top 20 most + 10 least vulnerable  |  min %d swings',
      542L
    ),
    x = NULL, y = 'Mean distortion share'
  )

options(repr.plot.width = 13, repr.plot.height = 10)
p_batter
ggsave('results/plots/applications/batter_vulnerability_overall.png', p_batter,
       width = 13, height = 10, dpi = 150)

In [ ]:
batter_top <- batter_pt |>
  filter(pitch_type %in% c('ST', 'CU', 'CH', 'SL')) |>
  mutate(
    batter_full_name = str_to_title(batter_full_name),
    pitch_name = factor(PITCH_NAMES[pitch_type],
                        levels = c('Sweeper', 'Curveball', 'Changeup', 'Slider'))
  ) |>
  group_by(pitch_type) |>
  arrange(desc(mean_distortion_share)) |>
  slice(1:8) |>
  ungroup()

p_bytype <- ggplot(batter_top,
                   aes(x = reorder(batter_full_name, mean_distortion_share),
                       y = mean_distortion_share,
                       fill = pitch_type)) +
  geom_col(width = 0.7) +
  scale_fill_manual(values = PITCH_COLORS, guide = 'none') +
  scale_y_continuous(labels = percent_format(accuracy = 1),
                     expand = expansion(mult = c(0.02, 0.15))) +
  facet_wrap(~ pitch_name, scales = 'free_y', ncol = 2) +
  coord_flip() +
  labs(
    title    = 'Batter Vulnerability by Pitch Type',
    subtitle = 'Top 8 most distortion-vulnerable batters for each of the four high-movement pitch types  |  min 50 swings',
    x = NULL, y = 'Mean distortion share'
  )

options(repr.plot.width = 14, repr.plot.height = 10)
p_bytype
ggsave('results/plots/applications/batter_vulnerability_by_pitch_type.png', p_bytype,
       width = 14, height = 10, dpi = 150)